# 第九階段:互動式 play 瀏覽器

**最終目標**:做一個「傳球空間分析」工具,給教練或球探使用。

**本階段只做一件事**:把第七階段的 play 卡包成一個**互動介面** —— 用下拉選單選一個 play,立刻看到那個 play 的 play 卡。這是從「筆記本分析」往「教練點一點就能用的工具」邁進的一步。

---

### 設計重點

- **可信優先**:根據第八階段的發現,passing lane 對深傳不可靠,所以 play 卡上會把第④格明確標註為**僅供參考**,避免誤導。separation 與 Voronoi 是主力。
- **重用,不重造**:直接沿用第七階段驗證過的 play 卡繪圖邏輯,只在外面包一層互動控制。
- **操作簡單**:一個下拉選單(列出所有傳球 play + 文字描述),選了就出卡。

> 註:互動元件需要在 **Jupyter 中實際執行**才會出現下拉選單。若只是靜態瀏覽這個 .ipynb,看到的是最後一次執行留下的那張卡。

## 步驟 0:載入工具

- **輸入**:無
- **輸出**:工具 + `ipywidgets`
- **目的**:`ipywidgets` 提供下拉選單等互動元件。

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.spatial import cKDTree
import ipywidgets as widgets
from IPython.display import display

plt.rcParams["font.sans-serif"] = ["PingFang TC", "Heiti TC", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

sys.path.append(str(Path.cwd().parent))
from config import DATA_DIR, SAMPLE_GAME_ID

print("gameId:", SAMPLE_GAME_ID, "/ ipywidgets", widgets.__version__)

gameId: 2021090900 / ipywidgets 8.1.9


## 步驟 1:載入資料 + play 卡繪圖邏輯(沿用第七階段)

- **輸入**:整場四個資料檔
- **輸出**:`prepare_play()`、四個 panel 函式、`play_card()`
- **目的**:這一格把第七階段的 play 卡程式整包搬過來(略作標註調整),讓互動介面可以直接呼叫。細節解說見 `07_play_card.ipynb`。

In [2]:
plays = pd.read_csv(DATA_DIR / "plays.csv")
players = pd.read_csv(DATA_DIR / "players.csv")
pff = pd.read_csv(DATA_DIR / "pffScoutingData.csv")
tracking = pd.read_csv(DATA_DIR / "tracking" / f"tracking_{SAMPLE_GAME_ID}.csv")

FIELD_X, FIELD_Y, STEP = 120.0, 53.3, 0.5
_gx = np.arange(0, FIELD_X, STEP) + STEP / 2
_gy = np.arange(0, FIELD_Y, STEP) + STEP / 2
_GX, _GY = np.meshgrid(_gx, _gy)
GRID = np.column_stack([_GX.ravel(), _GY.ravel()])


def prepare_play(play_id):
    pl = plays[(plays["gameId"] == SAMPLE_GAME_ID) & (plays["playId"] == play_id)]
    if pl.empty:
        return None
    pl = pl.iloc[0]
    pt = tracking[(tracking["gameId"] == SAMPLE_GAME_ID) & (tracking["playId"] == play_id)]
    sr = pt.loc[pt["event"] == "ball_snap", "frameId"]
    pr = pt.loc[pt["event"] == "pass_forward", "frameId"]
    if sr.empty or pr.empty:
        return None
    snap_frame, pass_frame = sr.iloc[0], pr.iloc[0]
    roles = pff[(pff["gameId"] == SAMPLE_GAME_ID) & (pff["playId"] == play_id)][["nflId", "pff_role"]]
    seg = (
        pt[(pt["frameId"] >= snap_frame) & (pt["frameId"] <= pass_frame)]
        .merge(players[["nflId", "displayName"]], on="nflId", how="left")
        .merge(roles, on="nflId", how="left")
    )
    return {"play_id": play_id, "info": pl, "off_team": pl["possessionTeam"],
            "def_team": pl["defensiveTeam"], "snap_frame": snap_frame,
            "pass_frame": pass_frame, "seg": seg}


def _p2s(px, py, ax0, ay0, bx, by):
    vx, vy = bx - ax0, by - ay0
    wx, wy = px - ax0, py - ay0
    L2 = vx * vx + vy * vy
    if L2 == 0:
        return np.hypot(px - ax0, py - ay0), 0.0
    t = (wx * vx + wy * vy) / L2
    tc = max(0.0, min(1.0, t))
    return np.hypot(px - (ax0 + tc * vx), py - (ay0 + tc * vy)), t


def _snapshot(d, ax):
    s = d["seg"][(d["seg"]["frameId"] == d["snap_frame"]) & d["seg"]["nflId"].notna()]
    o = s[s["team"] == d["off_team"]]; f = s[s["team"] == d["def_team"]]
    ax.scatter(o["x"], o["y"], c="tab:blue", s=60, label=f"進攻 {d['off_team']}")
    ax.scatter(f["x"], f["y"], c="tab:red", s=60, label=f"防守 {d['def_team']}")
    ax.set_xlim(0, FIELD_X); ax.set_ylim(0, FIELD_Y); ax.set_aspect("equal")
    ax.set_title("① 發球當下場上位置"); ax.legend(loc="upper right", fontsize=7)


def _voronoi(d, ax):
    s = d["seg"][(d["seg"]["frameId"] == d["snap_frame"]) & d["seg"]["nflId"].notna()]
    _, idx = cKDTree(s[["x", "y"]].to_numpy()).query(GRID)
    owner = s["team"].to_numpy()[idx]
    off_pct = (owner == d["off_team"]).sum() / len(owner) * 100
    ax.imshow((owner == d["def_team"]).astype(int).reshape(_GX.shape), origin="lower",
              extent=[0, FIELD_X, 0, FIELD_Y], cmap=ListedColormap(["#a6c8ff", "#ffb3b3"]),
              alpha=0.7, aspect="equal")
    ax.scatter(s[s["team"] == d["off_team"]]["x"], s[s["team"] == d["off_team"]]["y"], c="tab:blue", s=40)
    ax.scatter(s[s["team"] == d["def_team"]]["x"], s[s["team"] == d["def_team"]]["y"], c="tab:red", s=40)
    ax.set_xlim(0, FIELD_X); ax.set_ylim(0, FIELD_Y)
    ax.set_title(f"② Voronoi 控制區(進攻控制 {off_pct:.0f}%)")


def _separation(d, ax):
    rec = d["seg"][d["seg"]["pff_role"] == "Pass Route"]
    dfn = d["seg"][d["seg"]["pff_role"] == "Coverage"]
    for name, rg in rec.groupby("displayName"):
        xs, ys = [], []
        for fid, rrow in rg.groupby("frameId"):
            dd = dfn[dfn["frameId"] == fid][["x", "y"]].to_numpy()
            r0 = rrow.iloc[0]
            if len(dd):
                xs.append((fid - d["snap_frame"]) * 0.1)
                ys.append(np.sqrt((dd[:, 0] - r0["x"]) ** 2 + (dd[:, 1] - r0["y"]) ** 2).min())
        ax.plot(xs, ys, marker="o", markersize=2, label=name)
    ax.axvline((d["pass_frame"] - d["snap_frame"]) * 0.1, color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel("發球後秒數"); ax.set_ylabel("separation(碼)")
    ax.set_title("③ 接球員與最近防守者距離"); ax.legend(fontsize=6, loc="upper left"); ax.grid(True, alpha=0.3)


def _lanes(d, ax):
    fr = d["seg"][(d["seg"]["frameId"] == d["pass_frame"]) & d["seg"]["nflId"].notna()]
    qb = fr[fr["pff_role"] == "Pass"]
    if qb.empty:
        ax.set_title("④ 傳球路徑(此 play 無 QB 標記)"); return
    qb = qb.iloc[0]
    rec = fr[fr["pff_role"] == "Pass Route"]; dfn = fr[fr["pff_role"] == "Coverage"]
    for _, r in rec.iterrows():
        blk = [dist for _, dd in dfn.iterrows()
               for dist, t in [_p2s(dd["x"], dd["y"], qb["x"], qb["y"], r["x"], r["y"])] if 0 <= t <= 1]
        md = min(blk) if blk else np.nan
        color = "tab:green" if (np.isnan(md) or md >= 3) else ("orange" if md >= 1.5 else "tab:red")
        ax.plot([qb["x"], r["x"]], [qb["y"], r["y"]], "--", color=color, linewidth=1.3)
    ax.scatter(rec["x"], rec["y"], c="tab:blue", s=50, label="接球員")
    ax.scatter(dfn["x"], dfn["y"], c="tab:red", s=50, label="防守員")
    ax.scatter([qb["x"]], [qb["y"]], c="black", s=110, marker="*", label="QB")
    ax.set_xlim(0, FIELD_X); ax.set_ylim(0, FIELD_Y); ax.set_aspect("equal")
    # 第八階段發現:passing lane 對深傳不可靠,標註僅供參考
    ax.set_title("④ 傳球路徑(僅供參考:平面指標,對深傳不準)", color="dimgray")
    ax.legend(fontsize=7, loc="upper right")


def play_card(play_id):
    d = prepare_play(play_id)
    if d is None:
        print(f"playId={play_id} 不是傳球 play,無法製卡。"); return
    info = d["info"]
    fig, axes = plt.subplots(2, 2, figsize=(15, 9))
    _snapshot(d, axes[0, 0]); _voronoi(d, axes[0, 1])
    _separation(d, axes[1, 0]); _lanes(d, axes[1, 1])
    title = (f"Play 卡 — playId {play_id}  |  {d['off_team']}(進攻) vs {d['def_team']}(防守)"
             f"  |  第 {info.get('down','?')} 檔還差 {info.get('yardsToGo','?')} 碼"
             f"  |  結果 {info.get('passResult','?')}")
    fig.suptitle(title, fontsize=13, y=0.99)
    fig.text(0.5, 0.005, info.get("playDescription", ""), ha="center", fontsize=9, color="dimgray")
    fig.tight_layout(rect=[0, 0.02, 1, 0.97])
    plt.show()


print("play 卡邏輯已就緒。")

play 卡邏輯已就緒。


## 步驟 2:建立「傳球 play 清單」給下拉選單

- **輸入**:整場 `tracking`、`plays`
- **輸出**:`options`(每個傳球 play 一個選項:顯示文字 → playId)
- **目的**:下拉選單需要一份清單。只放**傳球 play**(有 snap + pass_forward),每個選項顯示 playId + 攻守 + 檔數 + 文字描述,讓教練好辨認。

In [3]:
pass_play_ids = []
for pid in sorted(tracking["playId"].unique()):
    pt = tracking[tracking["playId"] == pid]
    if (pt["event"] == "ball_snap").any() and (pt["event"] == "pass_forward").any():
        pass_play_ids.append(pid)

options = []
for pid in pass_play_ids:
    row = plays[(plays["gameId"] == SAMPLE_GAME_ID) & (plays["playId"] == pid)]
    if row.empty:
        continue
    row = row.iloc[0]
    desc = str(row.get("playDescription", ""))
    label = f"#{pid} | {row['possessionTeam']} vs {row['defensiveTeam']} | 第{row.get('down','?')}檔 | {desc[:50]}"
    options.append((label, pid))

print("可選的傳球 play 數:", len(options))
print("前 3 個選項範例:")
for lab, _ in options[:3]:
    print(" ", lab)

可選的傳球 play 數: 92
前 3 個選項範例:
  #97 | TB vs DAL | 第3檔 | (13:33) (Shotgun) T.Brady pass incomplete deep rig
  #137 | DAL vs TB | 第1檔 | (13:18) (Shotgun) D.Prescott pass deep left to A.C
  #187 | DAL vs TB | 第2檔 | (12:23) (Shotgun) D.Prescott pass short middle to 


## 步驟 3:組裝互動介面

- **輸入**:`options`、`play_card()`
- **輸出**:一個下拉選單 + 即時更新的 play 卡
- **目的**:把下拉選單綁定到 play 卡。選單一變,就重畫對應 play 的卡。

> **在 Jupyter 執行這一格**會看到下拉選單。選不同 play,下面的卡會立即更新。

In [4]:
dropdown = widgets.Dropdown(options=options, value=137 if 137 in pass_play_ids else options[0][1],
                            description="選 play:", layout=widgets.Layout(width="90%"),
                            style={"description_width": "initial"})
out = widgets.Output()

def _on_change(change):
    if change["type"] == "change" and change["name"] == "value":
        with out:
            out.clear_output(wait=True)
            play_card(change["new"])

dropdown.observe(_on_change)

# 先畫一次預設 play(靜態瀏覽時也看得到結果)
with out:
    play_card(dropdown.value)

display(dropdown, out)

Dropdown(description='選 play:', index=1, layout=Layout(width='90%'), options=(('#97 | TB vs DAL | 第3檔 | (13:33…

Output()

## 本階段完成了什麼

- 把第七階段的 play 卡包成**互動介面**:下拉選單選 play、即時出卡。
- 選單只列**傳球 play**,每項顯示攻守、檔數、文字描述,方便辨認。
- 依第八階段的誠實發現,把 play 卡第④格(passing lane)標註為**僅供參考(對深傳不準)**,避免誤導教練。
- 這是整個專案第一個「教練點一點就能用」的工具形態 —— 從分析筆記本,走到可操作的瀏覽器。

**使用方式**:在 Jupyter 執行到步驟 3,用下拉選單挑任一傳球 play,下方 play 卡會立即更新。

**下一階段(等你確認理解後再開始)**:
1. **批次匯出賽後報告**:一次把整場傳球 play 的卡存成圖檔 / PDF。
2. **擴充到多場**:下拉選單再加一層「選比賽」,逐場讀對應的 tracking 檔。
3. **把工具打包成獨立 app**(例如 Streamlit),不需要開 Jupyter 也能用。